# 03 PyTorch 系统模拟多维线性回归

这一节用 PyTorch 从头模拟一个多维线性回归任务。

你要重点理解：

- 多维线性回归的数学形式是什么。
- 输入 `X`、权重 `w`、偏置 `b`、预测值 `y_hat` 的 shape 分别是什么。
- 为什么 `nn.Linear(num_features, 1)` 可以表示多维线性回归。
- 训练循环里每一步到底在做什么。
- 怎么判断模型有没有学到真实规律。

## 1. 多维线性回归是什么

一维线性回归长这样：

$$y = wx + b$$

多维线性回归只是把一个特征 `x` 变成多个特征：

$$y = w_1x_1 + w_2x_2 + w_3x_3 + \cdots + w_nx_n + b$$

写成矩阵形式就是：

$$\hat{y} = Xw + b$$

人话解释：每个特征都有一个权重，模型会学习“哪个特征更重要”。

## 2. 环境准备

本节使用 CPU 即可。多维线性回归很小，不需要 GPU。

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cpu")

print("torch version:", torch.__version__)
print("device:", device)

## 3. 先设计一个真实规律

我们先人为规定一个真实函数，然后让模型通过数据自己学出来。

假设每个样本有 4 个特征：

$$y = 3x_1 - 2x_2 + 1.5x_3 + 0.5x_4 + 4 + noise$$

所以真实参数是：

```python
true_w = [3.0, -2.0, 1.5, 0.5]
true_b = 4.0
```

训练目标：让 PyTorch 模型学到接近这些真实参数的 `w` 和 `b`。

In [ ]:
num_samples = 1000
num_features = 4

true_w = torch.tensor([[3.0], [-2.0], [1.5], [0.5]])
true_b = torch.tensor([4.0])

print("true_w.shape:", true_w.shape)
print("true_b.shape:", true_b.shape)

这里故意把 `true_w` 写成 `[4, 1]`，而不是 `[4]`。

因为后面输入矩阵 `X.shape = [样本数, 特征数] = [1000, 4]`。

矩阵乘法时：

```text
[1000, 4] @ [4, 1] -> [1000, 1]
```

输出 `[1000, 1]` 表示 1000 个样本，每个样本预测 1 个数。

## 4. 生成模拟数据

我们用随机数生成输入特征 `X`，再用真实公式生成标签 `y`。

为了更接近真实数据，标签里加一点噪声 `noise`。

In [ ]:
X = torch.randn(num_samples, num_features)
noise = 0.3 * torch.randn(num_samples, 1)
y = X @ true_w + true_b + noise

print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("前 5 个样本的特征：\n", X[:5])
print("前 5 个样本的标签：\n", y[:5])

## 5. 封装成 Dataset 和 DataLoader

训练模型时，通常不会一次性把所有数据喂进去，而是分 batch 训练。

这里用：

- `TensorDataset(X, y)`：把特征和标签配成样本。
- `random_split`：划分训练集和测试集。
- `DataLoader`：按 batch 取数据。

In [ ]:
dataset = TensorDataset(X, y)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=generator
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

batch_X, batch_y = next(iter(train_loader))
print("训练集样本数:", len(train_dataset))
print("测试集样本数:", len(test_dataset))
print("batch_X.shape:", batch_X.shape)
print("batch_y.shape:", batch_y.shape)

## 6. 定义多维线性回归模型

PyTorch 里可以直接用：

```python
nn.Linear(in_features=4, out_features=1)
```

它表示：输入 4 个特征，输出 1 个预测值。

数学上就是：

$$\hat{y} = Xw + b$$

In [ ]:
model = nn.Linear(in_features=num_features, out_features=1).to(device)

print(model)
print("模型权重 shape:", model.weight.shape)
print("模型偏置 shape:", model.bias.shape)

注意：`nn.Linear` 里的权重 shape 是 `[out_features, in_features]`。

这里是：

```python
model.weight.shape = [1, 4]
```

而我们前面的 `true_w.shape = [4, 1]`。

所以最后比较参数时，需要看 `model.weight.T`，也就是转置后的形状 `[4, 1]`。

## 7. 定义损失函数和优化器

线性回归是回归任务，常用均方误差：

```python
nn.MSELoss()
```

优化器这里先用 SGD：

```python
torch.optim.SGD(model.parameters(), lr=0.05)
```

### 7.1 优化器是什么

优化器就是**负责更新模型参数的工具**。

模型里的参数，比如 `nn.Linear` 里的 `weight` 和 `bias`，一开始是随机初始化的。训练时我们希望它们一步步变好，让 loss 变小。

这里要分清楚三件事：

| 步骤 | 谁负责 | 人话解释 |
|---|---|---|
| 算预测值 | `model(batch_X)` | 用当前参数做预测 |
| 算梯度 | `loss.backward()` | 告诉每个参数应该往哪个方向改 |
| 改参数 | `optimizer.step()` | 优化器真正更新 `weight` 和 `bias` |

也就是说：

```text
loss.backward() 只负责算梯度
optimizer.step() 才负责改参数
```

### 7.2 SGD 的更新规则

`SGD` 是 Stochastic Gradient Descent，随机梯度下降。

最基础的理解是：

```text
新参数 = 旧参数 - 学习率 × 梯度
```

比如：

```text
旧参数 = 5
梯度 = 2
学习率 lr = 0.1

新参数 = 5 - 0.1 × 2 = 4.8
```

梯度表示 loss 增长最快的方向，所以我们要朝反方向走，才有机会让 loss 下降。

### 7.3 `torch.optim.SGD` 常用参数

| 参数 | 含义 | 初学者怎么理解 |
|---|---|---|
| `params` | 要更新哪些参数 | 通常写 `model.parameters()` |
| `lr` | 学习率 | 每次更新参数迈多大的步子 |
| `momentum` | 动量 | 让更新方向带一点惯性，后面再深入 |
| `weight_decay` | 权重衰减 | 一种正则化手段，用来抑制参数过大 |

本节先只用：

```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
```

`lr` 不是越大越好：

- 太大：参数更新步子太猛，loss 可能震荡甚至变大。
- 太小：参数更新太慢，训练很久还没学好。
- 合适：loss 稳定下降。

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

print("优化器:", optimizer)
print("优化器管理的参数组数量:", len(optimizer.param_groups))
print("学习率 lr:", optimizer.param_groups[0]["lr"])

## 8. 训练模型

标准训练循环：

```python
pred = model(batch_X)
loss = loss_fn(pred, batch_y)

optimizer.zero_grad()
loss.backward()
optimizer.step()
```

人话解释：

1. 用当前参数预测。
2. 计算预测值和真实值差多少。
3. 清空旧梯度。
4. 反向传播算新梯度。
5. 根据梯度更新参数。

三个容易混的函数：

| 写法 | 作用 | 为什么必须有 |
|---|---|---|
| `optimizer.zero_grad()` | 清空上一轮梯度 | PyTorch 默认会累加梯度，不清空会把旧梯度混进来 |
| `loss.backward()` | 计算当前 loss 对每个参数的梯度 | 这一步只算梯度，还没有更新参数 |
| `optimizer.step()` | 根据梯度更新参数 | 这一步才真正修改 `weight` 和 `bias` |

顺序不能乱：

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

如果忘了 `step()`，模型参数不会变；如果忘了 `zero_grad()`，梯度会越累越多。

In [ ]:
num_epochs = 30
train_losses = []

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        pred = model(batch_X)
        loss = loss_fn(pred, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = batch_X.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    avg_loss = total_loss / total_samples
    train_losses.append(avg_loss)

    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1:02d} | train loss = {avg_loss:.6f}")

## 9. 观察训练曲线

如果训练正常，loss 应该整体下降，然后逐渐稳定。

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(train_losses)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training loss")
plt.grid(True)
plt.show()

## 10. 在测试集上评估

测试时不需要计算梯度，所以要用：

```python
with torch.no_grad():
```

这能减少内存消耗，也能避免不小心把测试过程加入计算图。

In [ ]:
model.eval()
test_loss = 0.0
test_samples = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        pred = model(batch_X)
        loss = loss_fn(pred, batch_y)

        batch_size = batch_X.size(0)
        test_loss += loss.item() * batch_size
        test_samples += batch_size

test_loss = test_loss / test_samples
print(f"test MSE = {test_loss:.6f}")

## 11. 对比真实参数和学习到的参数

我们知道真实参数，所以可以直接对比模型有没有学对。

In [ ]:
learned_w = model.weight.detach().T
learned_b = model.bias.detach()

print("真实权重 true_w：\n", true_w)
print("模型学到的权重 learned_w：\n", learned_w)
print("权重误差 learned_w - true_w：\n", learned_w - true_w)

print("\n真实偏置 true_b:", true_b.item())
print("模型学到的偏置 learned_b:", learned_b.item())
print("偏置误差:", (learned_b - true_b).item())

## 12. 看预测效果

多维特征不好直接画出回归直线，但可以画：

- 横轴：真实值
- 纵轴：预测值

如果模型很好，点应该接近一条斜率为 1 的直线。

In [ ]:
all_preds = []
all_targets = []

model.eval()
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        pred = model(batch_X.to(device)).cpu()
        all_preds.append(pred)
        all_targets.append(batch_y)

all_preds = torch.cat(all_preds, dim=0)
all_targets = torch.cat(all_targets, dim=0)

plt.figure(figsize=(5, 5))
plt.scatter(all_targets.numpy(), all_preds.numpy(), s=12, alpha=0.7)
min_value = min(all_targets.min().item(), all_preds.min().item())
max_value = max(all_targets.max().item(), all_preds.max().item())
plt.plot([min_value, max_value], [min_value, max_value], color="red")
plt.xlabel("true y")
plt.ylabel("predicted y")
plt.title("True values vs predictions")
plt.grid(True)
plt.show()

## 13. 用模型预测一个新样本

训练好之后，就可以输入一个新的 4 维特征，让模型给出预测。

In [ ]:
new_X = torch.tensor([[1.0, 2.0, -1.0, 0.5]])

with torch.no_grad():
    pred_y = model(new_X)

true_y_without_noise = new_X @ true_w + true_b

print("new_X.shape:", new_X.shape)
print("模型预测 pred_y:", pred_y.item())
print("真实公式计算 true_y_without_noise:", true_y_without_noise.item())

## 14. 本节总结

多维线性回归的核心形状：

| 对象 | shape | 含义 |
|---|---|---|
| `X` | `[样本数, 特征数]` | 输入特征矩阵 |
| `w` | `[特征数, 1]` | 每个特征对应一个权重 |
| `b` | `[1]` | 偏置 |
| `y` | `[样本数, 1]` | 标签 |
| `nn.Linear(4, 1)` | 输入 4 维，输出 1 维 | 多维线性回归模型 |

训练后要检查：

1. loss 是否下降。
2. 测试集 MSE 是否合理。
3. 学到的 `w`、`b` 是否接近真实参数。
4. 预测值和真实值散点图是否接近红色参考线。

## 15. 小练习

请你自己改一改：

1. 把特征数从 4 改成 6，重新设计 `true_w`。
2. 把噪声从 `0.3` 改成 `1.0`，观察参数是否更难学准。
3. 把学习率 `lr=0.05` 改成 `0.5` 或 `0.005`，观察 loss 曲线变化。
4. 把优化器从 `SGD` 改成 `Adam`，比较收敛速度。
5. 尝试把训练集比例从 80% 改成 60%。